# Quick Experiment

Run a minimal ingestion and retrieval flow against sample documents.

In [1]:
from _bootstrap import setup_project_path

setup_project_path()

WindowsPath('C:/Users/min/Desktop/project-v2/retrieval-lab')

In [6]:
from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_retrieval_pipeline
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEERS

config = RetrievalConfig.from_env().with_overrides(chunk_size=250, chunk_overlap=20, top_k=3)
ingest_service, search_service = create_retrieval_pipeline(config)

In [7]:
documents = ingest_service.add_texts(
    texts=[item["text"] for item in SAMPLE_ENGINEERS],
    metadatas=[item["metadata"] for item in SAMPLE_ENGINEERS],
)
len(documents), documents[0]

(7,
 Document(metadata={'engineer_id': 'eng-001', 'grade': 'SENIOR', 'status': 'AVAILABLE', 'engineer_role': 'DEVELOPER', 'engineer_type': 'FULL_TIME'}, page_content='[기술스택]\nJava / Spring Boot / PostgreSQL / Redis / Docker\n\n[경력]\n8년차 백엔드 개발자.'))

In [14]:
query = "PL"
results = search_service.search(query, top_k=2)
[(round(item.score, 4), item.document.metadata, item.document.page_content) for item in results]

[(0.7964,
  {'engineer_id': 'eng-003',
   'grade': 'SENIOR',
   'status': 'AVAILABLE',
   'engineer_role': 'DEVELOPER',
   'engineer_type': 'FULL_TIME'},
  '[기술스택]\nReact / TypeScript / Redux / Recharts / Figma\n\n[경력]\n6년차 프론트엔드 개발자.'),
 (0.7955,
  {'engineer_id': 'eng-004',
   'grade': 'INTERMEDIATE',
   'status': 'AVAILABLE',
   'engineer_role': 'DEVELOPER',
   'engineer_type': 'FREELANCER'},
  '[기술스택]\nReact / JavaScript / Vue.js / Chart.js / CSS\n\n[경력]\n4년차 프론트엔드 개발자.\n\n[프로젝트 경험]\n포스코(2024~2025): 생산 현황 모니터링 대시보드 개발. Chart.js 활용. 포지션: 프론트엔드 개발자.\n네이버(2022~2024): 광고 성과 리포트 화면 개발. 포지션: 프론트엔드 개발자.\n에이전시(2021~2022): 다수 기업 웹사이트 퍼블리싱. 포지션: 퍼블리셔.')]

In [16]:
# 벡터 DB 전체 초기화
ingest_service._vector_store._store.delete(delete_all=True)
print("done")

done
